In [7]:
%%writefile task1_tokens
import re
samples = ["AI, search!", "I can't log in.", "help@example.com",
"یہ ایک مثال ہے" ,"" ,"2.1 version 4944CS "]
for text in samples:
 ws = text.split()
 words = re.findall(r"\b\w+\b", text, flags=re.UNICODE)
 print(repr(text), "whitespace:", ws, len(ws))
 print("word-like:", words, len(words))
assert re.findall(r"\b\w+\b", "") == []


Writing task1_tokens


In [2]:
%%writefile lab6_search.py
from sklearn.feature_extraction.text import TfidfVectorizer

class Retriever:
    def __init__(self, records, backend="tfidf"):
        self.records = records
        self.vectorizer = TfidfVectorizer()
        texts = [r["text"] for r in records]
        self.matrix = self.vectorizer.fit_transform(texts)

    def search(self, query, k=3):
        if not query.strip():
            return []

        q = self.vectorizer.transform([query])

        if q.nnz == 0:
            return []

        scores = (q @ self.matrix.T).toarray()[0]
        ranked = sorted(
            range(len(scores)),
            key=lambda i: -scores[i]
        )

        return [
            (self.records[i], float(scores[i]))
            for i in ranked[:k]
        ]

Writing lab6_search.py


In [3]:
%%writefile guided_chatbot.py
from lab6_search import Retriever
FAQ = [
    {"id":"faq:password", "question":"How do I reset my password?",
 "answer":"Use the recovery link on the login page.",
 "source":"it_help.txt"},
    {"id":"faq:submit", "question":"Where do I submit my assignment?",
 "answer":"Use the assignment upload page in the learning portal.",
 "source":"assessment.txt"},
    {"id":"faq:late", "question":"Can illness justify a late submission?",
 "answer":"Request an extension with medical evidence before the deadline.",
 "source":"handbook.txt"},
    {"id":"faq:library", "question":"How can I access library resources?",
     "answer":"Visit the university library portal.",
     "source":"library.txt"},

    {"id":"faq:help", "question":"How can I contact student support?",
     "answer":"Contact the student support department.",
     "source":"support.txt"}]
def build_bot(backend="tfidf"):
    records = [dict(r, text=r["question"]+" "+r["answer"]) for r in FAQ]
    return Retriever(records, backend)
def reply(engine, query):
    if not query.strip():
        return "Please enter a question."
    for r in FAQ:
        if query.strip().casefold() == r["question"].casefold():
            return r["answer"] + " [source: " + r["source"] + "]"
    hits = engine.search(query)
    if not hits:
        return "No matching FAQ. Please contact the help desk."
    lines = ["Possible matches; choose or rephrase your question:"]
    for r,score in hits:
        lines.append(r["question"] + f" (similarity {score:.3f})")
    return "\n".join(lines)

Writing guided_chatbot.py


In [5]:
tests = [
    "How do I reset my password?",
    "password recovery",
    "",
    "volcano geology",
    "How can I access library resources?"
]

for question in tests:
    print("\nYou:", question)
    print("Bot:", reply(engine, question))


You: How do I reset my password?
Bot: Use the recovery link on the login page. [source: it_help.txt]

You: password recovery
Bot: Possible matches; choose or rephrase your question:
How do I reset my password? (similarity 0.448)
Where do I submit my assignment? (similarity 0.000)
Can illness justify a late submission? (similarity 0.000)

You: 
Bot: Please enter a question.

You: volcano geology
Bot: No matching FAQ. Please contact the help desk.

You: How can I access library resources?
Bot: Visit the university library portal. [source: library.txt]


What is TF-IDF?

TF-IDF measures the importance of words and helps find relevant FAQ questions.

Why does the chatbot suggest instead of answering paraphrased questions?

Because similar questions may have different meanings. Suggestions help prevent incorrect answers.

Why are sources displayed?

Sources help users identify where the answer came from.

In [8]:
!ls


guided_chatbot.py  __pycache__	task1_tokens
lab6_search.py	   sample_data	task1_tokens.py
